# Spatial Analysis: Geographic Patterns of Heat Stress

This notebook analyzes spatial patterns and geographic variations in heat stress metrics:

**Analysis Types:**
1. State-level choropleth maps
2. Regional comparisons (Great Plains, Southwest, etc.)
3. Latitude/longitude gradients
4. State ranking and percentiles
5. Spatial correlation analysis
6. Geographic clusters
7. Urban vs rural patterns (grid-level)
8. Elevation effects

**Geographic Focus:** Continental US with emphasis on cattle-producing regions

In [ ]:
# Standard imports
import xarray as xr
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.colors import LinearSegmentedColormap, BoundaryNorm
import seaborn as sns
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from pathlib import Path
from scipy.spatial.distance import pdist, squareform
from scipy.cluster.hierarchy import dendrogram, linkage
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.figsize'] = (14, 10)
plt.rcParams['font.size'] = 11
sns.set_style('whitegrid')

print("Spatial analysis imports complete!")

In [ ]:
# Configuration
DATA_DIR = Path('../..')  # Up two levels from notebooks/XX_category/ to research/  # Up one level from tests/ to research/
NIGHTTIME_DIR = DATA_DIR / 'processed_nighttime_recovery'
DAYTIME_DIR = DATA_DIR / 'processed_daytime_heat'
VPD_DIR = DATA_DIR / 'processed_vpd'
MASK_FILE = DATA_DIR / '../../masks/region_mask.nc'
OUTPUT_DIR = Path('../../figures/spatial')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# US state IDs and names from region_mask.nc (alphabetical order, NOT standard FIPS)
STATE_NAMES = {
    1: 'Alabama', 2: 'Arkansas', 3: 'Arizona', 4: 'California', 5: 'Colorado',
    6: 'Connecticut', 7: 'Delaware', 8: 'Florida', 9: 'Georgia', 10: 'Iowa',
    11: 'Idaho', 12: 'Illinois', 13: 'Indiana', 14: 'Kansas', 15: 'Kentucky',
    16: 'Louisiana', 17: 'Massachusetts', 18: 'Maryland', 19: 'Maine', 20: 'Michigan',
    21: 'Minnesota', 22: 'Missouri', 23: 'Mississippi', 24: 'Montana', 25: 'North Carolina',
    26: 'North Dakota', 27: 'Nebraska', 28: 'New Hampshire', 29: 'New Jersey', 30: 'New Mexico',
    31: 'Nevada', 32: 'New York', 33: 'Ohio', 34: 'Oklahoma', 35: 'Oregon',
    36: 'Pennsylvania', 37: 'Rhode Island', 38: 'South Carolina', 39: 'South Dakota', 40: 'Tennessee',
    41: 'Texas', 42: 'Utah', 43: 'Virginia', 44: 'Vermont', 45: 'Washington',
    46: 'Wisconsin', 47: 'West Virginia', 48: 'Wyoming'
}

# Focus states (Regions 4 & 6: major cattle-producing states)
FOCUS_STATES = {
    'Alabama': 1,
    'Arizona': 3,
    'Florida': 8,
    'Georgia': 9,
    'Kentucky': 15,
    'Louisiana': 16,
    'Mississippi': 23,
    'New Mexico': 30,
    'North Carolina': 25,
    'Oklahoma': 34,
    'South Carolina': 38,
    'Tennessee': 40,
    'Texas': 41
}

# Regional groupings (using correct IDs from alphabetical ordering)
REGIONS = {
    'Great Plains': [14, 27, 26, 39],  # Kansas, Nebraska, North Dakota, South Dakota
    'Southern Plains': [34, 41],  # Oklahoma, Texas
    'Southwest': [3, 30],  # Arizona, New Mexico
    'Mountain West': [5, 11, 24, 42, 48],  # Colorado, Idaho, Montana, Utah, Wyoming
    'Corn Belt': [12, 13, 10, 22],  # Illinois, Indiana, Iowa, Missouri
    'Southeast': [1, 8, 9, 23, 38],  # Alabama, Florida, Georgia, Mississippi, South Carolina
    'West Coast': [4, 35, 45]  # California, Oregon, Washington
}

print(f"Data directory: {DATA_DIR.resolve()}")
print(f"Output directory: {OUTPUT_DIR.resolve()}")

In [ ]:
# Load state mask
ds_mask = xr.open_dataset(MASK_FILE)
state_mask = ds_mask.state_mask
lat = ds_mask.lat.values
lon = ds_mask.lon.values
ds_mask.close()

# Helper functions
def load_monthly_data(data_dir, year_start=2010, year_end=2023):
    """Load monthly data for a given period."""
    files = sorted(data_dir.glob('*.nc'))
    datasets = []
    for f in files:
        year_month = f.stem.split('_')[-1]
        year = int(year_month[:4])
        if year_start <= year <= year_end:
            datasets.append(xr.open_dataset(f))
    if datasets:
        return xr.concat(datasets, dim='time')
    return None

def compute_state_stats(data, state_ids=None):
    """Compute mean for each state.
    
    Converts timedelta64[ns] to float hours if needed.
    """
    if state_ids is None:
        state_ids = STATE_NAMES.keys()
    
    state_means = {}
    for state_id in state_ids:
        mask = state_mask == state_id
        masked_data = data.where(mask)
        state_mean = masked_data.mean(dim=['time', 'lat', 'lon'])
        
        # Convert timedelta to hours if data is stored as timedelta
        if state_mean.dtype.kind == 'm':  # 'm' = timedelta
            # Convert nanoseconds to hours: divide by (3600 * 10^9)
            state_mean = state_mean / np.timedelta64(1, 'h')
            state_mean = state_mean.astype(np.float64)
        
        if not np.isnan(state_mean.values):
            state_means[state_id] = float(state_mean.values)
    
    return state_means

# Load data (recent period for detailed analysis)
print("Loading data (2010-2023)...")
ds_night = load_monthly_data(NIGHTTIME_DIR, 2010, 2023)
ds_day = load_monthly_data(DAYTIME_DIR, 2010, 2023)
ds_vpd = load_monthly_data(VPD_DIR, 2010, 2023)
print("Data loaded!")

## Plot 1: State-Level Choropleth Maps

Geographic distribution of heat stress metrics across US states.

In [ ]:
# Compute summer averages for each state
metrics_to_map = [
    ('hours_above_24', 'Nighttime Hours Above 24°C\n(Poor Recovery)', ds_night, 'YlOrRd'),
    ('hours_above_35', 'Daytime Hours Above 35°C\n(Extreme Heat)', ds_day, 'RdPu'),
    ('vpd_mean', 'Afternoon Mean VPD (kPa)', ds_vpd, 'YlGnBu')
]

fig, axes = plt.subplots(3, 1, figsize=(16, 18), 
                        subplot_kw={'projection': ccrs.PlateCarree()})

for idx, (metric, title, ds, cmap) in enumerate(metrics_to_map):
    ax = axes[idx]
    
    # Filter for summer months
    summer_data = ds[metric].where(ds.time.dt.month.isin([6, 7, 8]), drop=True)
    
    # Compute state means
    state_stats = compute_state_stats(summer_data)
    
    # Create map data
    map_data = np.full_like(state_mask.values, np.nan, dtype=float)
    for state_id, value in state_stats.items():
        map_data[state_mask.values == state_id] = value
    
    # Plot
    im = ax.pcolormesh(lon, lat, map_data, transform=ccrs.PlateCarree(),
                       cmap=cmap, shading='auto')
    
    ax.coastlines()
    ax.add_feature(cfeature.BORDERS, linestyle=':')
    ax.add_feature(cfeature.STATES, linewidth=0.5, edgecolor='black')
    ax.set_extent([-125, -66, 24, 49], crs=ccrs.PlateCarree())
    
    # Colorbar
    cbar = plt.colorbar(im, ax=ax, orientation='horizontal', pad=0.05, aspect=40)
    cbar.set_label(title, fontsize=12, fontweight='bold')
    
    ax.set_title(f'{title}\nSummer Average (2010-2023)', fontweight='bold', fontsize=14)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / '01_state_choropleth_maps.png', dpi=300, bbox_inches='tight')
plt.show()
print("Plot 1 saved!")

## Plot 2: Regional Comparison Boxplots

Compare distributions across major livestock regions.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
axes = axes.flatten()

metrics_regional = [
    ('hours_above_24', 'Nighttime Hours > 24°C', ds_night),
    ('hours_above_30', 'Daytime Hours > 30°C', ds_day),
    ('vpd_mean', 'Mean VPD (kPa)', ds_vpd),
    ('hours_below_0', 'Nighttime Hours < 0°C', ds_night)
]

for idx, (metric, title, ds) in enumerate(metrics_regional):
    ax = axes[idx]
    
    # Filter for summer (or winter for cold metrics)
    if 'below' in metric:
        season_data = ds[metric].where(ds.time.dt.month.isin([12, 1, 2]), drop=True)
    else:
        season_data = ds[metric].where(ds.time.dt.month.isin([6, 7, 8]), drop=True)
    
    # Collect data by region
    data_by_region = []
    labels = []
    
    for region_name, state_ids in REGIONS.items():
        region_values = []
        for state_id in state_ids:
            mask = state_mask == state_id
            state_data = season_data.where(mask).values.flatten()
            
            # Convert timedelta to hours if needed
            if state_data.dtype.kind == 'm':  # 'm' = timedelta
                state_data = state_data / np.timedelta64(1, 'h')
            
            state_data = state_data[~np.isnan(state_data)]
            region_values.extend(state_data)
        
        if len(region_values) > 0:
            data_by_region.append(region_values)
            labels.append(region_name)
    
    # Create boxplot
    bp = ax.boxplot(data_by_region, labels=labels, patch_artist=True,
                    showmeans=True, meanline=True)
    
    # Color boxes
    colors = plt.cm.Set3(np.linspace(0, 1, len(labels)))
    for patch, color in zip(bp['boxes'], colors):
        patch.set_facecolor(color)
        patch.set_alpha(0.7)
    
    ax.set_xticklabels(labels, rotation=45, ha='right')
    ax.set_ylabel(title)
    ax.set_title(f'Regional Comparison: {title}', fontweight='bold')
    ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig(OUTPUT_DIR / '02_regional_boxplots.png', dpi=300, bbox_inches='tight')
plt.show()
print("Plot 2 saved!")

## Plot 3: Latitude and Longitude Gradients

How heat stress varies with latitude and longitude.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 10))

metric = 'hours_above_30'
summer_data = ds_day[metric].where(ds_day.time.dt.month.isin([6, 7, 8]), drop=True)
mean_data = summer_data.mean(dim='time')

# Convert timedelta to hours if needed
if mean_data.dtype.kind == 'm':  # 'm' = timedelta
    mean_data = mean_data / np.timedelta64(1, 'h')
    mean_data = mean_data.astype(np.float64)

# Latitude gradient (averaged over longitude)
ax = axes[0, 0]
lat_profile = mean_data.mean(dim='lon')
ax.plot(lat_profile.lat, lat_profile.values, linewidth=3, color='darkred')
ax.fill_between(lat_profile.lat, lat_profile.values, alpha=0.3, color='red')
ax.set_xlabel('Latitude (°N)')
ax.set_ylabel('Hours Above 30°C')
ax.set_title('North-South Gradient\n(Zonal Mean)', fontweight='bold')
ax.grid(True, alpha=0.3)

# Longitude gradient (averaged over latitude)
ax = axes[0, 1]
lon_profile = mean_data.mean(dim='lat')
ax.plot(lon_profile.lon, lon_profile.values, linewidth=3, color='darkblue')
ax.fill_between(lon_profile.lon, lon_profile.values, alpha=0.3, color='blue')
ax.set_xlabel('Longitude (°E)')
ax.set_ylabel('Hours Above 30°C')
ax.set_title('East-West Gradient\n(Meridional Mean)', fontweight='bold')
ax.grid(True, alpha=0.3)

# Scatter: Latitude vs Heat Stress
ax = axes[1, 0]
# Create scatter data
lats_2d, lons_2d = np.meshgrid(lat, lon, indexing='ij')
mask_us = state_mask.values > 0
scatter_lat = lats_2d[mask_us]
scatter_values = mean_data.values[mask_us]
valid = ~np.isnan(scatter_values)
ax.scatter(scatter_lat[valid], scatter_values[valid], alpha=0.3, s=5, c=scatter_values[valid],
          cmap='YlOrRd')
# Add trend line
z = np.polyfit(scatter_lat[valid], scatter_values[valid], 2)
p = np.poly1d(z)
lat_range = np.linspace(scatter_lat[valid].min(), scatter_lat[valid].max(), 100)
ax.plot(lat_range, p(lat_range), 'k--', linewidth=2, label='Quadratic fit')
ax.set_xlabel('Latitude (°N)')
ax.set_ylabel('Hours Above 30°C')
ax.set_title('Latitude vs Heat Stress\n(All Grid Cells)', fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3)

# Scatter: Longitude vs Heat Stress
ax = axes[1, 1]
scatter_lon = lons_2d[mask_us]
ax.scatter(scatter_lon[valid], scatter_values[valid], alpha=0.3, s=5, c=scatter_values[valid],
          cmap='YlOrRd')
z = np.polyfit(scatter_lon[valid], scatter_values[valid], 2)
p = np.poly1d(z)
lon_range = np.linspace(scatter_lon[valid].min(), scatter_lon[valid].max(), 100)
ax.plot(lon_range, p(lon_range), 'k--', linewidth=2, label='Quadratic fit')
ax.set_xlabel('Longitude (°E)')
ax.set_ylabel('Hours Above 30°C')
ax.set_title('Longitude vs Heat Stress\n(All Grid Cells)', fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / '03_geographic_gradients.png', dpi=300, bbox_inches='tight')
plt.show()
print("Plot 3 saved!")

## Plot 4: State Rankings

Top and bottom states for various heat stress metrics.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 14))
axes = axes.flatten()

metrics_ranking = [
    ('hours_above_24', 'Worst Nighttime Recovery\n(Hours > 24°C)', ds_night, True),
    ('hours_above_35', 'Most Extreme Daytime Heat\n(Hours > 35°C)', ds_day, True),
    ('vpd_mean', 'Highest Afternoon VPD (kPa)', ds_vpd, True),
    ('hours_below_0', 'Most Freezing Nights\n(Hours < 0°C)', ds_night, True)
]

for idx, (metric, title, ds, ascending) in enumerate(metrics_ranking):
    ax = axes[idx]
    
    # Filter for appropriate season
    if 'below' in metric:
        season_data = ds[metric].where(ds.time.dt.month.isin([12, 1, 2]), drop=True)
    else:
        season_data = ds[metric].where(ds.time.dt.month.isin([6, 7, 8]), drop=True)
    
    # Compute state means
    state_stats = compute_state_stats(season_data)
    
    # Sort and get top 15
    sorted_states = sorted(state_stats.items(), key=lambda x: x[1], reverse=not ascending)
    top_states = sorted_states[:15] if ascending else sorted_states[-15:]
    
    # Prepare data for plotting
    state_names = [STATE_NAMES.get(s[0], f'State {s[0]}') for s in top_states]
    values = [s[1] for s in top_states]
    
    # Create horizontal bar chart
    colors = plt.cm.RdYlGn_r(np.linspace(0.3, 0.9, len(values)))
    bars = ax.barh(range(len(state_names)), values, color=colors)
    
    ax.set_yticks(range(len(state_names)))
    ax.set_yticklabels(state_names, fontsize=10)
    ax.set_xlabel(metric.replace('_', ' ').title())
    ax.set_title(title, fontweight='bold', fontsize=12)
    ax.grid(True, alpha=0.3, axis='x')
    
    # Add value labels
    for i, (bar, val) in enumerate(zip(bars, values)):
        ax.text(val + max(values)*0.01, i, f'{val:.1f}',
                ha='left', va='center', fontsize=9)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / '04_state_rankings.png', dpi=300, bbox_inches='tight')
plt.show()
print("Plot 4 saved!")

## Plot 5: Spatial Correlation Heatmap

How heat stress metrics correlate across states.

In [ ]:
# Select key states
key_states = {
    'TX': 48, 'OK': 40, 'KS': 20, 'NE': 31, 'SD': 46,
    'MT': 30, 'CA': 6, 'AZ': 4, 'FL': 12, 'GA': 13
}

metric = 'hours_above_30'
summer_data = ds_day[metric].where(ds_day.time.dt.month.isin([6, 7, 8]), drop=True)

# Compute daily time series for each state
state_timeseries = {}
for state_abbr, state_id in key_states.items():
    mask = state_mask == state_id
    state_data = summer_data.where(mask).mean(dim=['lat', 'lon'])
    
    # Convert timedelta to hours if needed
    if state_data.dtype.kind == 'm':  # 'm' = timedelta
        state_data = state_data / np.timedelta64(1, 'h')
    
    state_timeseries[state_abbr] = state_data.values

# Create DataFrame
df = pd.DataFrame(state_timeseries)

# Compute correlation matrix
corr_matrix = df.corr()

# Plot heatmap
fig, ax = plt.subplots(figsize=(12, 10))
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='RdBu_r',
            center=0, vmin=-1, vmax=1, square=True,
            linewidths=1, cbar_kws={'label': 'Correlation'})
ax.set_title('Spatial Correlation of Daytime Heat\n(Hours > 30°C, Summer 2010-2023)',
            fontweight='bold', fontsize=14)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / '05_spatial_correlation.png', dpi=300, bbox_inches='tight')
plt.show()
print("Plot 5 saved!")

## Plot 6: Hierarchical Clustering Dendrogram

Group states by similarity in heat stress patterns.

In [ ]:
# Use the correlation matrix from above
# Convert correlation to distance
distance_matrix = 1 - corr_matrix.abs()

# Perform hierarchical clustering
linkage_matrix = linkage(squareform(distance_matrix), method='average')

# Plot dendrogram
fig, ax = plt.subplots(figsize=(14, 8))
dendrogram(linkage_matrix, labels=corr_matrix.columns, ax=ax,
          color_threshold=0.5, above_threshold_color='gray')
ax.set_xlabel('State', fontsize=12, fontweight='bold')
ax.set_ylabel('Distance (1 - |Correlation|)', fontsize=12, fontweight='bold')
ax.set_title('Hierarchical Clustering of States\nBased on Heat Stress Patterns',
            fontweight='bold', fontsize=14)
ax.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / '06_clustering_dendrogram.png', dpi=300, bbox_inches='tight')
plt.show()
print("Plot 6 saved!")

## Plot 7: Grid-Level Variability Maps

Spatial standard deviation showing heterogeneous vs homogeneous regions.

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(16, 14),
                        subplot_kw={'projection': ccrs.PlateCarree()})

metrics_variability = [
    ('hours_above_30', 'Daytime Hours > 30°C\nTemporal Variability (Std Dev)', ds_day),
    ('vpd_mean', 'Mean VPD\nTemporal Variability (Std Dev)', ds_vpd)
]

for idx, (metric, title, ds) in enumerate(metrics_variability):
    ax = axes[idx]
    
    # Filter for summer
    summer_data = ds[metric].where(ds.time.dt.month.isin([6, 7, 8]), drop=True)
    
    # Convert timedelta to hours if needed (before computing std)
    if summer_data.dtype.kind == 'm':  # 'm' = timedelta
        summer_data = summer_data / np.timedelta64(1, 'h')
    
    # Compute standard deviation over time
    std_data = summer_data.std(dim='time')
    
    # Mask non-US areas
    std_data_masked = std_data.where(state_mask > 0)
    
    # Plot
    im = ax.pcolormesh(lon, lat, std_data_masked, transform=ccrs.PlateCarree(),
                      cmap='viridis', shading='auto')
    
    ax.coastlines()
    ax.add_feature(cfeature.BORDERS, linestyle=':')
    ax.add_feature(cfeature.STATES, linewidth=0.5, edgecolor='black')
    ax.set_extent([-125, -66, 24, 49], crs=ccrs.PlateCarree())
    
    cbar = plt.colorbar(im, ax=ax, orientation='horizontal', pad=0.05, aspect=40)
    cbar.set_label(title, fontsize=12, fontweight='bold')
    
    ax.set_title(f'{title}\n(2010-2023)', fontweight='bold', fontsize=14)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / '07_variability_maps.png', dpi=300, bbox_inches='tight')
plt.show()
print("Plot 7 saved!")

## Plot 8: Regional Aggregate Comparison

Radar chart comparing regions across multiple metrics.

In [ ]:
# Select metrics for radar chart
radar_metrics = [
    ('hours_above_24', ds_night, 'Poor Recovery'),
    ('hours_above_30', ds_day, 'Very Hot Days'),
    ('hours_above_35', ds_day, 'Extreme Heat'),
    ('vpd_mean', ds_vpd, 'High VPD'),
    ('hours_below_0', ds_night, 'Freezing Nights')
]

# Collect data for selected regions
selected_regions = ['Great Plains', 'Southern Plains', 'Southwest', 'Southeast']
region_data = {region: [] for region in selected_regions}

for metric, ds, label in radar_metrics:
    # Filter for appropriate season
    if 'below' in metric:
        season_data = ds[metric].where(ds.time.dt.month.isin([12, 1, 2]), drop=True)
    else:
        season_data = ds[metric].where(ds.time.dt.month.isin([6, 7, 8]), drop=True)
    
    # Compute for each region
    for region_name in selected_regions:
        state_ids = REGIONS[region_name]
        region_values = []
        for state_id in state_ids:
            mask = state_mask == state_id
            state_mean = season_data.where(mask).mean()
            
            # Convert timedelta to hours if needed
            if state_mean.dtype.kind == 'm':  # 'm' = timedelta
                state_mean = state_mean / np.timedelta64(1, 'h')
            
            if not np.isnan(state_mean.values):
                region_values.append(float(state_mean.values))
        
        if region_values:
            region_data[region_name].append(np.mean(region_values))
        else:
            region_data[region_name].append(0)

# Normalize data to 0-1 scale for each metric
normalized_data = {region: [] for region in selected_regions}
for i in range(len(radar_metrics)):
    values = [region_data[region][i] for region in selected_regions]
    min_val, max_val = min(values), max(values)
    for region in selected_regions:
        if max_val > min_val:
            normalized = (region_data[region][i] - min_val) / (max_val - min_val)
        else:
            normalized = 0.5
        normalized_data[region].append(normalized)

# Create radar chart
labels = [label for _, _, label in radar_metrics]
num_vars = len(labels)
angles = np.linspace(0, 2 * np.pi, num_vars, endpoint=False).tolist()
angles += angles[:1]  # Complete the circle

fig, ax = plt.subplots(figsize=(10, 10), subplot_kw=dict(projection='polar'))

colors = ['red', 'orange', 'purple', 'green']
for region, color in zip(selected_regions, colors):
    values = normalized_data[region]
    values += values[:1]  # Complete the circle
    ax.plot(angles, values, 'o-', linewidth=2, label=region, color=color)
    ax.fill(angles, values, alpha=0.15, color=color)

ax.set_xticks(angles[:-1])
ax.set_xticklabels(labels, size=11)
ax.set_ylim(0, 1)
ax.set_yticks([0.2, 0.4, 0.6, 0.8, 1.0])
ax.set_yticklabels(['20%', '40%', '60%', '80%', '100%'], size=9)
ax.grid(True)
ax.legend(loc='upper right', bbox_to_anchor=(1.3, 1.1), fontsize=11)
ax.set_title('Regional Heat Stress Profile\n(Normalized, 2010-2023)',
            fontweight='bold', fontsize=14, pad=20)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / '08_regional_radar_chart.png', dpi=300, bbox_inches='tight')
plt.show()
print("Plot 8 saved!")
print("\n=== Spatial Analysis Complete ===")
print(f"8 plots saved to {OUTPUT_DIR.resolve()}")

## Summary

This notebook generated 8 comprehensive spatial analysis plots:

1. **State Choropleth Maps** - Geographic distribution across US
2. **Regional Boxplots** - Compare major livestock production regions
3. **Geographic Gradients** - Latitude/longitude effects on heat stress
4. **State Rankings** - Top/bottom states for each metric
5. **Spatial Correlation** - How heat stress co-varies across states
6. **Clustering Dendrogram** - States grouped by similarity
7. **Variability Maps** - Spatial heterogeneity in heat stress
8. **Regional Radar Chart** - Multi-metric regional comparison

**Key Findings:**
- Southern Plains (TX, OK) have highest heat stress
- Strong latitude gradient: heat decreases northward
- Coastal areas show lower VPD than interior
- Great Plains states cluster together (similar patterns)
- High spatial variability in Southwest (topography effects)